## 🏗️ LeetCode 56: Merge Intervals
---

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #e3f2fd; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> Sort by start time. Then a single pass is sufficient: if the current interval's start is ≤ the last merged interval's end, they overlap — extend the last merged interval's end to max(both ends). If not, they don't overlap — add current as a new merged interval.
</div>

### 🛠️ The Mathematical Model

After sorting by start, two adjacent intervals overlap if and only if `intervals[i][0] <= merged[-1][1]`. When they overlap, the merged end is `max(merged[-1][1], intervals[i][1])`.

$$\text{overlap condition: } start_i \leq end_{last} \Rightarrow end_{merged} = \max(end_{last}, end_i)$$

---

### 📋 Problem

Given an array of intervals where `intervals[i] = [start_i, end_i]`, merge all overlapping intervals and return the resulting array of non-overlapping intervals.

**Example 1:**
```
Input:  [[1,3],[2,6],[8,10],[15,18]]
Output: [[1,6],[8,10],[15,18]]
```

**Example 2:**
```
Input:  [[1,4],[4,5]]
Output: [[1,5]]
```

**Constraints:** 1 ≤ intervals.length ≤ 10⁴ | 0 ≤ start_i ≤ end_i ≤ 10⁴

### 🧠 Choose Your Mental Model

<table style="width:100%; border-collapse: collapse;">
    <tr style="background-color: #f2f2f2; text-align: left;">
        <th style="padding: 12px; border: 1px solid #ddd;">Model</th>
        <th style="padding: 12px; border: 1px solid #ddd;">The "Story"</th>
        <th style="padding: 12px; border: 1px solid #ddd;">Mechanism</th>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Timeline Painter</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"I'm painting a timeline. Each interval is a stroke. Sort strokes by start. If the next stroke starts before my current stroke ends, extend my current stroke. If it starts after, finish the current stroke and start a new one."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Track `last_end`. Extend if `start <= last_end`. Otherwise append new interval.</td>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Booking Consolidator</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"Consolidate overlapping hotel bookings into continuous stays. Sort by check-in. If next guest checks in before current guest checks out, merge into one stay (checkout = latest of both)."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Sort by start + max(ends) merging = contiguous stay consolidation</td>
    </tr>
</table>

---

### ⚠️ Performance Warning

<div style="padding: 10px; border: 1px solid #ffe58f; background-color: #fffbe6; border-radius: 4px;">
    <strong>Note:</strong> Naive pairwise comparison checks all O(n²) pairs. After sorting once (O(n log n)), a single linear pass merges in O(n). Total: O(n log n) — the sort dominates.
</div>

## 🐢 Approach 1: Brute Force — $O(n^2)$

In [ ]:
def merge_brute(intervals):
    """
    Brute Force: Repeatedly scan for any pair that overlaps and merge them
    Time: O(n^2) | Space: O(n)
    """
    intervals = [list(i) for i in intervals]
    changed = True
    while changed:
        changed = False
        for i in range(len(intervals)):
            for j in range(i + 1, len(intervals)):
                a, b = intervals[i], intervals[j]
                # Overlap check: [a0,a1] and [b0,b1] overlap if a0<=b1 and b0<=a1
                if a[0] <= b[1] and b[0] <= a[1]:
                    intervals[i] = [min(a[0], b[0]), max(a[1], b[1])]
                    intervals.pop(j)
                    changed = True
                    break
            if changed:
                break
    return sorted(intervals)


print(merge_brute([[1,3],[2,6],[8,10],[15,18]]))   # Expected: [[1,6],[8,10],[15,18]]

## 🔬 Complexity Analysis: $O(n^2)$ vs. $O(n \log n)$

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #f0f7ff; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> After sorting by start time, overlapping intervals are always adjacent. We never need to check non-adjacent pairs. A single linear pass from left to right processes each interval exactly once — O(n) after the O(n log n) sort.
</div>

---

## 📉 Why Brute Force Fails: The $O(n^2)$ Trap

Pairwise comparison: for n = 10,000 intervals, that's 50 million pair checks. Each merge requires another scan from the beginning.

| Input Type | Brute Force Performance | Reason |
|------------|------------------------|--------|
| **All overlapping** | O(n²) merges | Each merge triggers full rescan |
| **Already sorted** | O(n²) pair checks | Still checks all pairs |

---

## 🚀 The Optimal Approach: $O(n \log n)$

Sort by start time. Maintain a `merged` result list. For each interval:
- If it overlaps with the last merged (start ≤ last_end), extend last merged's end
- Otherwise, append as a new merged interval

### The Key Lifecycle Rule
1. **Sort by start time** — ensures overlapping intervals are adjacent
2. **Initialize merged** with the first interval
3. **For each subsequent interval:** extend if overlap, append if not

---

## ✅ Mathematical Proof

After sorting, if intervals i and j overlap (i < j), there is no non-overlapping interval k between them (otherwise j's start > k's end > i's end, contradiction). So all overlapping intervals form contiguous groups — a single pass processes each group.

<div style="padding: 10px; border-left: 8px solid #4CAF50; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>✅ Summary:</strong> Sorting transforms a 2D comparison problem into a 1D scan. After sorting by start, all overlapping intervals are adjacent — one pass with max(ends) merging is sufficient.
</div>

## 🚀 Approach 2: Sort + Linear Merge — $O(n \log n)$
---

Instead of pairwise comparison, we use **sort once, scan once**.

As we iterate:
1. Sort intervals by start time
2. Initialize merged = [intervals[0]]
3. For each interval: if start ≤ merged[-1][1], extend. Else append.

In [ ]:
def merge(intervals: list[list[int]]) -> list[list[int]]:
    """
    Optimal: Sort by start, single linear merge pass
    Time: O(n log n) | Space: O(n) for output
    """
    intervals.sort(key=lambda x: x[0])   # sort by start time
    merged = [intervals[0]]

    for start, end in intervals[1:]:
        last_end = merged[-1][1]
        if start <= last_end:           # overlap — extend
            merged[-1][1] = max(last_end, end)   # critical: max, not just end
        else:                           # no overlap — new interval
            merged.append([start, end])

    return merged


print("Optimal:", merge([[1,3],[2,6],[8,10],[15,18]]))   # Expected: [[1,6],[8,10],[15,18]]
print("Optimal:", merge([[1,4],[4,5]]))                   # Expected: [[1,5]]
print("Optimal:", merge([[1,4],[2,3]]))                   # Expected: [[1,4]] (containment)

## 🎤 5 Interview Q&A

### Q1: What is the time complexity and why?

**Answer:** O(n log n) — dominated by the sort. The subsequent linear scan is O(n). For n = 10,000 intervals: sort takes ~130,000 comparisons, scan takes 10,000. Total = O(n log n).

---

### Q2: Why `max(last_end, end)` and not just `end`?

**Answer:** One interval can completely contain another — e.g., [1,10] followed by [2,5] in sorted order. Without max, processing [2,5] after [1,10] would shrink the merged end to 5, losing the [5,10] portion. `max(last_end, end)` preserves the farthest endpoint seen.

---

### Q3: What is the time complexity of the optimal approach and why?

**Answer:** O(n log n). The sort is O(n log n) — this dominates. The merge pass is O(n) — each interval is processed exactly once. Overall: O(n log n + n) = O(n log n).

---

### Q4: How would you solve this if intervals were already sorted?

**Answer:** Skip the sort, go straight to the merge pass: O(n). Initialize merged = [intervals[0]], then the same extend/append logic. The sort is the only thing that makes the naive O(n²) problem solvable in O(n) — if it's free, we get O(n).

---

### Q5: What are the edge cases to watch for?

**Answer:** (1) Single interval — for loop never executes, merged = [[start, end]], returned directly. (2) All overlapping — all intervals merge into one. (3) No overlapping (already disjoint) — each interval appended as-is, output = sorted input. (4) One interval containing another — [1,10] then [2,5]: max(10, 5) = 10, correctly extends. (5) Adjacent touching — [1,4],[4,5]: start=4 ≤ last_end=4, merged. Problem says [4,5] as example.

## 📚 Key Terminology

| Term | Definition |
|------|------------|
| **Interval** | A range `[start, end]` representing a contiguous segment of values or time |
| **Overlap** | Two intervals `[a,b]` and `[c,d]` overlap if `a ≤ d AND c ≤ b` — they share at least one point |
| **Merge** | Combining two overlapping intervals into one: `[min(starts), max(ends)]` |
| **Sort by Start** | The key preprocessing step — makes overlapping intervals adjacent in the sorted order |
| **Max(ends)** | Critical in merge: accounts for containment (one interval fully inside another) |
| **Containment** | When interval B is fully inside interval A — A contains B. max(ends) handles this case. |
| **Disjoint Intervals** | Non-overlapping intervals — the output of merge() is always disjoint |
| **Linear Scan** | Single O(n) pass after sorting — processes each interval exactly once |

## 💼 The Citi Narrative

**Context:** Citi's capacity planning team tracked server maintenance windows — time ranges when servers were taken offline for patching. For compliance reporting, they needed to report total downtime hours per server per quarter, without double-counting overlapping windows.

**Scenario:** A server might have three maintenance windows in a quarter: [09:00-11:00], [10:30-12:30], [15:00-16:00]. Naive sum = 5.5 hours. Correct merged: [[09:00-12:30],[15:00-16:00]] = 4.5 hours. Merge Intervals was the algorithm.

**How this pattern applied:** For 6,000 servers × multiple maintenance windows each quarter, sorting windows by start time and then linear merge pass produced accurate per-server downtime in O(n log n). The alternative — pairwise overlap checks — was O(n²) per server.

**Impact:** Quarterly downtime compliance reports for 6,000 servers produced in seconds instead of hours. More importantly, accurate merged downtime caught double-counting errors in the previous manual process — the real total downtime was 15% lower than previously reported, which changed SLA compliance calculations.

In [ ]:
# -------------------------------------------------------
# PRACTICE: Given a list of intervals representing employee
# working hours, find the free time (gaps between working hours).
# E.g. [[1,3],[6,7],[9,12]] → free time: [[3,6],[7,9]]
# -------------------------------------------------------

def findFreeTime(intervals):
    intervals.sort(key=lambda x: x[0])
    # Your solution here — find gaps between merged intervals
    pass


# Test
print(findFreeTime([[1,3],[6,7],[9,12]]))     # Expected: [[3,6],[7,9]]
print(findFreeTime([[1,3],[2,4],[5,6]]))      # Expected: [[4,5]]

## 🎯 Summary: Key Takeaways

### The Pattern
**Sort + Linear Merge** — Sort by start time; one pass with max(ends) merging handles all overlap cases

### When to Use It
- ✅ Merging overlapping time ranges, windows, bookings, events
- ✅ Computing total covered time without double-counting
- ✅ Preprocessing for other interval operations (insert interval, etc.)
- ❌ **Don't use when:** Intervals are streaming (need different approach — interval tree or sorted container)

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force (pairwise) | $O(n^2)$ | $O(n)$ |
| Optimal (Sort + Scan) | $O(n \log n)$ | $O(n)$ |

### Interview Confidence Checklist
- [ ] Can state the overlap condition from memory: `a ≤ d AND c ≤ b`
- [ ] Can explain why max(ends) is needed (containment case)
- [ ] Can write the solution from memory in 4 minutes
- [ ] Can reduce to O(n) when input is pre-sorted
- [ ] Can connect to maintenance window / downtime reporting use case

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master Merge Intervals and you've mastered the foundational interval technique — sort once, scan once, handle all cases. 🚀